# Office-Altformat-Konverter

Konvertiert alte Office-Formate in moderne Formate:
- `.doc` → `.docx` (via Word)
- `.ppt` → `.pptx` (via PowerPoint)
- `.xls` → `.xlsx` (via Excel)

**Voraussetzung:** Microsoft Office + `pywin32`

In [2]:
try:
    import win32com.client
    print("✅ pywin32 ist installiert.")
except ImportError:
    print("pywin32 wird installiert...")
    %pip install pywin32 -q
    import win32com.client
    print("✅ pywin32 installiert.")

✅ pywin32 ist installiert.


In [3]:
import os
import shutil
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown

# ════════════════════════════════════════════════════════════════
# Konfiguration — hier anpassen
# ════════════════════════════════════════════════════════════════

ORDNER_PFAD = r"D:\03_LehramtT"   # ← Deinen Pfad hier eintragen
BACKUP = True                       # True = Originale sichern

IGNORIERTE_VERZEICHNISSE = {
    "node_modules", ".git", "__pycache__", ".ipynb_checkpoints",
    ".venv", "venv", "env", ".env", "_office_backup",
}

print(f"Ordner:  {ORDNER_PFAD}")
print(f"Backup:  {'ja → _office_backup/' if BACKUP else 'nein'}")

Ordner:  D:\03_LehramtT
Backup:  ja → _office_backup/


## 1. Altformat-Dateien finden

In [4]:
def altformat_finden(basis_pfad: str) -> dict[str, list[Path]]:
    """Findet rekursiv alle .doc, .ppt, .xls Dateien."""
    basis = Path(basis_pfad).resolve()
    gefunden = {"doc": [], "ppt": [], "xls": []}
    endungen = {".doc": "doc", ".ppt": "ppt", ".xls": "xls"}

    for verz, unterverz, dateien in os.walk(basis):
        unterverz[:] = [
            d for d in unterverz
            if d not in IGNORIERTE_VERZEICHNISSE and not d.startswith(".")
        ]
        for name in sorted(dateien):
            nl = name.lower()
            for endung, typ in endungen.items():
                if nl.endswith(endung) and not nl.endswith(endung + "x"):
                    gefunden[typ].append(Path(verz) / name)

    return gefunden


# Suchen
gefunden = altformat_finden(ORDNER_PFAD)

# Ergebnis anzeigen
labels = {"doc": "Word (.doc → .docx)", "ppt": "PowerPoint (.ppt → .pptx)", "xls": "Excel (.xls → .xlsx)"}
gesamt = 0

for typ in ["doc", "ppt", "xls"]:
    dateien = gefunden[typ]
    gesamt += len(dateien)
    if dateien:
        print(f"\n📄 {labels[typ]}: {len(dateien)} Dateien")
        for d in dateien[:20]:
            kb = d.stat().st_size / 1024
            print(f"   {kb:7.0f} KB  {d}")
        if len(dateien) > 20:
            print(f"   ... und {len(dateien) - 20} weitere")

if gesamt == 0:
    print("✅ Keine Altformat-Dateien gefunden.")
else:
    print(f"\n{'='*50}")
    print(f"Gesamt: {gesamt} Dateien zum Konvertieren")
    print(f"{'='*50}")


📄 Word (.doc → .docx): 60 Dateien
        76 KB  D:\03_LehramtT\01_Che\Chemie_08\01_ErsteChemiestunde\VerhaltenimChemieraum.doc
        44 KB  D:\03_LehramtT\01_Che\Chemie_08\02_PraktikumLebensmittel\ErsteChemiestundeAB_MusterlsgNeueVersion.doc
        73 KB  D:\03_LehramtT\01_Che\Chemie_08\13_Gemische\Stoffgemischebenennen.doc
       151 KB  D:\03_LehramtT\01_Che\Chemie_08\33_GesetzErhaltungDerMasse\000GesetzvonderErhaltungderMasse.doc
        38 KB  D:\03_LehramtT\01_Che\Chemie_08\33_GesetzErhaltungDerMasse\Front33.doc
        48 KB  D:\03_LehramtT\01_Che\Chemie_08\34_KonstanteMassenverhaeltnisse\000GesetzvondenkonstantenProportionen.doc
       106 KB  D:\03_LehramtT\01_Che\Chemie_08\34_KonstanteMassenverhaeltnisse\ABGesetzvondenkonstantenMassenverhaeltnissen.doc
        38 KB  D:\03_LehramtT\01_Che\Chemie_08\34_KonstanteMassenverhaeltnisse\Front34.doc
        33 KB  D:\03_LehramtT\01_Che\Chemie_08\35_Dalton\000DasDaltonAtommodell.doc
        26 KB  D:\03_LehramtT\01_Che\Chemie_08\3

## 2. Konvertieren

Die nächste Zelle startet Word, PowerPoint und Excel im Hintergrund und konvertiert alle gefundenen Dateien. **Pro Datei 1–3 Sekunden.**

⚠️ Während der Konvertierung die Office-Anwendungen bitte nicht manuell öffnen.

In [5]:
def _backup_verschieben(quelle: Path, backup_ordner: Path):
    """Verschiebt Original in Backup-Ordner."""
    if not backup_ordner:
        return
    backup_ordner.mkdir(parents=True, exist_ok=True)
    ziel = backup_ordner / quelle.name
    if ziel.exists():
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        ziel = backup_ordner / f"{quelle.stem}_{ts}{quelle.suffix}"
    shutil.move(str(quelle), str(ziel))


def konvertiere_word(dateien: list[Path], backup_ordner: Path = None) -> list[dict]:
    if not dateien:
        return []
    print(f"─── Word: {len(dateien)} Dateien ───")
    ergebnisse = []
    word = None
    try:
        word = win32com.client.Dispatch("Word.Application")
        word.Visible = False
        word.DisplayAlerts = False
        for i, quelle in enumerate(dateien, 1):
            ziel = quelle.with_suffix(".docx")
            print(f"  [{i}/{len(dateien)}] {quelle.name}", end=" ")
            if ziel.exists():
                print("⏭️ .docx existiert")
                ergebnisse.append({"quelle": str(quelle), "status": "übersprungen", "typ": "doc"})
                continue
            try:
                doc = word.Documents.Open(str(quelle))
                doc.SaveAs2(str(ziel), FileFormat=16)
                doc.Close()
                print("✅")
                ergebnisse.append({"quelle": str(quelle), "ziel": str(ziel), "status": "ok", "typ": "doc"})
                _backup_verschieben(quelle, backup_ordner)
            except Exception as ex:
                print(f"❌ {ex}")
                ergebnisse.append({"quelle": str(quelle), "status": "fehler", "grund": str(ex)[:120], "typ": "doc"})
    finally:
        if word:
            try: word.Quit()
            except: pass
    return ergebnisse


def konvertiere_powerpoint(dateien: list[Path], backup_ordner: Path = None) -> list[dict]:
    if not dateien:
        return []
    print(f"─── PowerPoint: {len(dateien)} Dateien ───")
    ergebnisse = []
    ppt_app = None
    try:
        ppt_app = win32com.client.Dispatch("PowerPoint.Application")
        for i, quelle in enumerate(dateien, 1):
            ziel = quelle.with_suffix(".pptx")
            print(f"  [{i}/{len(dateien)}] {quelle.name}", end=" ")
            if ziel.exists():
                print("⏭️ .pptx existiert")
                ergebnisse.append({"quelle": str(quelle), "status": "übersprungen", "typ": "ppt"})
                continue
            try:
                prs = ppt_app.Presentations.Open(str(quelle), WithWindow=False)
                prs.SaveAs(str(ziel), FileFormat=24)
                prs.Close()
                print("✅")
                ergebnisse.append({"quelle": str(quelle), "ziel": str(ziel), "status": "ok", "typ": "ppt"})
                _backup_verschieben(quelle, backup_ordner)
            except Exception as ex:
                print(f"❌ {ex}")
                ergebnisse.append({"quelle": str(quelle), "status": "fehler", "grund": str(ex)[:120], "typ": "ppt"})
    finally:
        if ppt_app:
            try: ppt_app.Quit()
            except: pass
    return ergebnisse


def konvertiere_excel(dateien: list[Path], backup_ordner: Path = None) -> list[dict]:
    if not dateien:
        return []
    print(f"─── Excel: {len(dateien)} Dateien ───")
    ergebnisse = []
    excel = None
    try:
        excel = win32com.client.Dispatch("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        for i, quelle in enumerate(dateien, 1):
            ziel = quelle.with_suffix(".xlsx")
            print(f"  [{i}/{len(dateien)}] {quelle.name}", end=" ")
            if ziel.exists():
                print("⏭️ .xlsx existiert")
                ergebnisse.append({"quelle": str(quelle), "status": "übersprungen", "typ": "xls"})
                continue
            try:
                wb = excel.Workbooks.Open(str(quelle))
                wb.SaveAs(str(ziel), FileFormat=51)
                wb.Close()
                print("✅")
                ergebnisse.append({"quelle": str(quelle), "ziel": str(ziel), "status": "ok", "typ": "xls"})
                _backup_verschieben(quelle, backup_ordner)
            except Exception as ex:
                print(f"❌ {ex}")
                ergebnisse.append({"quelle": str(quelle), "status": "fehler", "grund": str(ex)[:120], "typ": "xls"})
    finally:
        if excel:
            try: excel.Quit()
            except: pass
    return ergebnisse


# ─── Ausführung ────────────────────────────────────────────

basis = Path(ORDNER_PFAD).resolve()
backup_ordner = (basis / "_office_backup") if BACKUP else None

if backup_ordner:
    print(f"📦 Backup → {backup_ordner}\n")

alle = []
alle.extend(konvertiere_word(gefunden["doc"], backup_ordner))
alle.extend(konvertiere_powerpoint(gefunden["ppt"], backup_ordner))
alle.extend(konvertiere_excel(gefunden["xls"], backup_ordner))

# ─── Zusammenfassung ──────────────────────────────────────

ok = sum(1 for e in alle if e["status"] == "ok")
fehler = sum(1 for e in alle if e["status"] == "fehler")
skip = sum(1 for e in alle if e["status"] == "übersprungen")

print(f"\n{'='*50}")
print(f"✅ Konvertiert:  {ok}")
print(f"⏭️ Übersprungen: {skip}")
print(f"❌ Fehler:       {fehler}")
print(f"{'='*50}")

if fehler > 0:
    print("\nFehlgeschlagene Dateien:")
    for e in alle:
        if e["status"] == "fehler":
            print(f"  {e['quelle']}")
            print(f"    → {e.get('grund', '?')}")

📦 Backup → D:\03_LehramtT\_office_backup

─── Word: 60 Dateien ───
  [1/60] VerhaltenimChemieraum.doc ✅
  [2/60] ErsteChemiestundeAB_MusterlsgNeueVersion.doc ✅
  [3/60] Stoffgemischebenennen.doc ✅
  [4/60] 000GesetzvonderErhaltungderMasse.doc ✅
  [5/60] Front33.doc ✅
  [6/60] 000GesetzvondenkonstantenProportionen.doc ✅
  [7/60] ABGesetzvondenkonstantenMassenverhaeltnissen.doc ✅
  [8/60] Front34.doc ✅
  [9/60] 000DasDaltonAtommodell.doc ✅
  [10/60] Front35.doc ✅
  [11/60] ABGenauigkeitundsignifikanteStellenvonMesswerten.doc ✅
  [12/60] Front36.doc ✅
  [13/60] UeBzuratomareMasseneinheit.doc ✅
  [14/60] 000AnzahlverhaeltnisvonAtomenbeichemoschenReaktionen.doc ✅
  [15/60] Front37.doc ✅
  [16/60] ABMolekuele.doc ✅
  [17/60] VomReaktionsschemazurReaktionsgleichung.doc ✅
  [18/60] VomReaktionsschemazurReaktionsgleichungLsg.doc ✅
  [19/60] Front40.doc ✅
  [20/60] LoesungenundUebungAnzahlverhaeltnisundElementargruppe.doc ✅
  [21/60] 000TeilchenanzahlundStoffmenge.doc ✅
  [22/60] Front41.doc ✅
 

## 3. Ergebnis prüfen

Kurzer Check: Sind noch `.doc`/`.ppt`/`.xls`-Dateien übrig?

In [6]:
rest = altformat_finden(ORDNER_PFAD)
rest_gesamt = sum(len(v) for v in rest.values())

if rest_gesamt == 0:
    print("✅ Keine Altformat-Dateien mehr vorhanden. Alles konvertiert!")
else:
    print(f"⚠️ Noch {rest_gesamt} Altformat-Dateien übrig:\n")
    for typ in ["doc", "ppt", "xls"]:
        for d in rest[typ]:
            print(f"  .{typ} → {d}")

⚠️ Noch 1 Altformat-Dateien übrig:

  .xls → D:\03_LehramtT\04_Nwt\NWT_08\Bruckenbau\Tab_zentrales_kraeftesystem.xls
